# 🛡️TIF Análisis de Detección de Intrusiones en Red

**Grupo 1**: Gonza Gabriela · Casasola Hernán · Biazutti Luciano · Lera Aníbal Iván · Alvarado Marcelo

**Módulo**: Ciencias de Datos y Optimización de Modelos

**Carrera**: Tecnicatura Universitaria en Ciencias de Datos e IA Aplicada — UPATECO

---



En esta notebook continuamos el trabajo iniciado en proyecto_eda.ipynb.  
El objetivo es construir una muestra representativa del dataset para trabajar el entrenamiento.

In [1]:
# Librerías generales
import io
import warnings
import requests
import numpy as np
import pandas as pd
from google.colab import files

warnings.filterwarnings('ignore')

In [2]:
# Configuración
GITHUB_USER = 'jotaeleb'
GITHUB_REPO = 'tif-ciencias-de-datos'
GITHUB_BRANCH = 'main'
DATASET_DIR = 'dataset'

BASE_URL = (
    f'https://raw.githubusercontent.com/'
    f'{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{DATASET_DIR}'
)

PARQUET_FILES = [
    'dataset_parte_1.parquet',
    'dataset_parte_2.parquet',
    'dataset_parte_3.parquet',
    'dataset_parte_4.parquet',
]

# Carga de datasets
dfs = [
    pd.read_parquet(
        io.BytesIO(requests.get(f'{BASE_URL}/{file}').content)
    )
    for file in PARQUET_FILES
]

df = pd.concat(dfs, ignore_index=True)

# Limpieza básica
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].str.replace(r'[^\w\s-]', '-', regex=True)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

# Variables constantes (varianza = 0)
rango = df[num_cols].max() - df[num_cols].min()
constantes = rango[rango == 0].index.tolist()

# Columnas a eliminar
cols_duplicadas = [
    'Fwd Header Length.1',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'SYN Flag Count',
    'CWE Flag Count'
]

cols_corruptas = [
    'Fwd Header Length',
    'Bwd Header Length'
]

to_drop = list(set(constantes + cols_duplicadas + cols_corruptas))

# Dataset limpio
df_clean = df.drop(columns=to_drop)

# Variables
num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
cat_cols = df_clean.select_dtypes(exclude=np.number).columns.tolist()

# Resumen
print(f'Dataset original: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Dataset limpio: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas')
print(f'Columnas eliminadas: {len(to_drop)}')
print(f'Variables numéricas: {len(num_cols)} | Variables categóricas: {len(cat_cols)}')

Dataset original: 2,827,876 filas × 79 columnas
Dataset limpio: 2,827,876 filas × 64 columnas
Columnas eliminadas: 15
Variables numéricas: 63 | Variables categóricas: 1


In [3]:
# Peso del archivo
peso_mb = df_clean.memory_usage(deep=True).sum() / 1024**2
print(f'DataFrame: {peso_mb:.2f} MB')

DataFrame: 1508.30 MB


In [4]:
# Pasar a .parquet
df_clean.to_parquet('df_muestra.parquet', index=False)

In [5]:
# Descarga
files.download('df_muestra.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>